# 🧩 **00_setup_environment.ipynb — Full Content**

## 🎯 **Result: Notebook 00 Is Fully Complete**

This notebook:

- checks all required packages  
- generates `utils.py`  
- generates `pqc_libs.json`  
- generates `benchmarks.json`  
- is Fedora‑compatible  
- is robust and minimalistic  
- forms the foundation for all other notebooks  
- fits perfectly into our hybrid architecture  

In [2]:
## 📌 Section 1 — Imports & Helpers
import os
import importlib
import json
from datetime import datetime

def exists(path):
    return os.path.exists(path)

def check_package(pkg):
    try:
        importlib.import_module(pkg)
        print(f"✓ {pkg} found")
        return True
    except ImportError:
        print(f"✗ {pkg} missing — install via terminal:")
        print(f"    pip install {pkg}")
        return False

In [4]:
import sys
!{sys.executable} -m pip install cmake git+https://github.com/open-quantum-safe/liboqs-python.git
!{sys.executable} -m pip install fastapi uvicorn

  Cloning https://github.com/open-quantum-safe/liboqs-python.git to /tmp/pip-req-build-m59c8opq
  Running command git clone --filter=blob:none --quiet https://github.com/open-quantum-safe/liboqs-python.git /tmp/pip-req-build-m59c8opq
  Resolved https://github.com/open-quantum-safe/liboqs-python.git to commit cbf1788acdecf5f54c3b29900c7e404d618616cb
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [fastapi]m2/3 [fastapi]


Execute in jupyter terminal:
```bash
sudo dnf install -y cmake gcc gcc-c++ ninja-build git openssl-devel make
```

In [5]:
## 📌 Section 2 — Package Check

required = [
    "oqs",          # PQC algorithms
    "cryptography", # ECDSA
    "pandas",       # CSV handling
    "matplotlib",   # plotting
    "fastapi",      # service demo
    "uvicorn",      # service runner
    "aiohttp",      # client
    "requests"      # HTTP client
]

print("=== Package Check ===")
missing = [pkg for pkg in required if not check_package(pkg)]

if missing:
    print("\nMissing packages detected.")
    print("Install them in your Jupyter terminal:")
    print("pip install " + " ".join(missing))
else:
    print("\nAll required packages available.")

=== Package Check ===
✓ oqs found
✓ cryptography found
✓ pandas found
✓ matplotlib found
✓ fastapi found
✓ uvicorn found
✓ aiohttp found
✓ requests found

All required packages available.


Execute in a terminal:
```bash
echo "$HOME/_oqs/lib64" | sudo tee /etc/ld.so.conf.d/liboqs.conf
sudo ldconfig
ldconfig -p | grep liboqs
```

In [6]:
import oqs
print(oqs.oqs_version())

0.16.0


In [7]:
## 📌 Section 3 — Generate `utils.py` Module  
# This module is imported by all other notebooks.

utils_code = """
import time
import matplotlib.pyplot as plt

def timer(fn, *args, **kwargs):
    start = time.perf_counter()
    result = fn(*args, **kwargs)
    end = time.perf_counter()
    return result, end - start

def plot_bar(labels, values, title, path):
    plt.figure(figsize=(6,4))
    plt.bar(labels, values)
    plt.title(title)
    plt.ylabel("Value")
    plt.savefig(path)
    plt.close()

def plot_line(labels, values, title, path):
    plt.figure(figsize=(6,4))
    plt.plot(labels, values, marker='o')
    plt.title(title)
    plt.ylabel("Value")
    plt.savefig(path)
    plt.close()
"""

os.makedirs("../modules", exist_ok=True)

with open("../modules/utils.py", "w") as f:
    f.write(utils_code)

print("✓ Created modules/utils.py")


✓ Created modules/utils.py


In [9]:
## 📌 Section 4 — PQC Library Check & JSON Export

import oqs

info = {
    "timestamp": datetime.now().isoformat(),
    "oqs_version": oqs.oqs_version(),
    "oqs_python_version": oqs.oqs_python_version(),
    "available_algorithms": {
        "signatures": oqs.get_enabled_sig_mechanisms(),
        "kem": oqs.get_enabled_kem_mechanisms()
    }
}

os.makedirs("../data", exist_ok=True)

with open("../data/pqc_libs.json", "w") as f:
    json.dump(info, f, indent=2)

print("✓ Created data/pqc_libs.json")

✓ Created data/pqc_libs.json


In [13]:
## 📌 Section 5 — Mini‑Benchmark (Keygen / Sign / Verify)

import sys, os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import oqs
from modules.utils import timer

def benchmark_signature(alg):
    msg = b"benchmark"

    with oqs.Signature(alg) as sig:
        # Keygen (returns the public key)
        public_key, t_keygen = timer(sig.generate_keypair)

        # Sign
        signature, t_sign = timer(sig.sign, msg)

        # Verify
        _, t_verify = timer(sig.verify, msg, signature, public_key)

    return {
        "algorithm": alg,
        "keygen": t_keygen,
        "sign": t_sign,
        "verify": t_verify
    }

enabled_sigs = oqs.get_enabled_sig_mechanisms()

algorithms = ["ML-DSA-65", "Falcon-512"]

results = []
for alg in algorithms:
    if alg not in enabled_sigs:
        raise ValueError(
            f"'{alg}' is not enabled in this liboqs build.\n"
            f"Available signature mechanisms:\n{enabled_sigs}"
        )
    print(f"Running benchmark for {alg}...")
    results.append(benchmark_signature(alg))

df = pd.DataFrame(results)
os.makedirs("../data", exist_ok=True)
df.to_json("../data/benchmarks.json", orient="records", indent=2)

print("✓ Created data/benchmarks.json")

Running benchmark for ML-DSA-65...
Running benchmark for Falcon-512...
✓ Created data/benchmarks.json


In [14]:
## 📌 Section 6 — Completion Message

print("\n=== Setup Complete ===")
print("utils.py, pqc_libs.json, benchmarks.json generated.")
print("You can now continue with Notebook 01.")


=== Setup Complete ===
utils.py, pqc_libs.json, benchmarks.json generated.
You can now continue with Notebook 01.
